**Etapa 1 — Entender a carteira**

**Objetivo:** descobrir quem são os clientes da base.

**Perguntas:**

Quantos clientes existem?
Qual a distribuição por idade?
Como os clientes estão distribuídos geograficamente?
Existe concentração em determinadas regiões?
Qual a distribuição por gênero?
Qual o saldo médio?
Qual o valor médio das transações?
Qual a frequência média de utilização?


**É o diagnóstico da carteira.**



In [0]:
# Manipulação e estruturação de dados
import pandas as pd
import numpy as np

# Visualização de dados
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning e Métricas (caso vá avançar para modelagem preditiva)
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

In [0]:
df = spark.sql("""
    SELECT *
    FROM bank_transactions
""").toPandas()

df.head()

**Quantos clientes existem?**

In [0]:
%sql
select count(CustomerID) as total_clientes from bank_transactions


In [0]:
df.info()
df.describe()
df.isnull().sum()

In [0]:
(df.isnull().sum() / len(df) * 100).round(2)

Muito saudável para uma base de mais de 1 milhão de registros.

Alterando os tipos de data para as correta!

In [0]:
df.rename(columns={
    "TransactionID": "ID_Transacao",
    "CustomerID": "ID_Cliente",
    "CustomerDOB": "Data_Nascimento",
    "CustGender": "Genero",
    "CustLocation": "Localizacao",
    "CustAccountBalance": "Saldo_Conta",
    "TransactionDate": "Data_Transacao",
    "TransactionTime": "Horario_Transacao",
    "TransactionAmount (INR)": "Valor_Transacao"
}, inplace=True)

In [0]:
###COnvertendo datas

df['Data_Nascimento'] = pd.to_datetime(
    df['Data_Nascimento'],
    dayfirst=True,
    errors='coerce'
)

df['Data_Transacao'] = pd.to_datetime(
    df['Data_Transacao'],
    dayfirst=True,
    errors='coerce'
)

### _Ajustando_ "consumerDOB - Data_Nascimento " para uma nova coluna "AGE - IDADE" para ter uma idade atualizada

In [0]:
%sql
SELECT distinct CustomerDOB
from bank_transactions

In [0]:
# 1. Trata 1800-01-01 como data inválida
df.loc[
    df["Data_Nascimento"] == pd.Timestamp("1800-01-01"),
    "Data_Nascimento"
] = pd.NaT

# 2. Corrige anos futuros/incompatíveis
df.loc[
    df["Data_Nascimento"].dt.year > 2026,
    "Data_Nascimento"
] = df.loc[
    df["Data_Nascimento"].dt.year > 2026,
    "Data_Nascimento"
].apply(lambda x: x.replace(year=x.year - 100))

# 3. Calcula idade
df["Idade"] = 2026 - df["Data_Nascimento"].dt.year

In [0]:
df["Idade"].describe()

In [0]:
print("Menores de 18:", (df["Idade"] < 18).sum())
print("18 a 89:", ((df["Idade"] >= 18) & (df["Idade"] < 90)).sum())
print("90 ou mais:", (df["Idade"] >= 90).sum())
print("Nulos:", df["Idade"].isna().sum())

In [0]:
df.loc[(df["Idade"] < 18) | (df["Idade"] > 100), "Idade"] = None
df[["Data_Nascimento", "Idade"]].head(20)

Agrupando "Idade " por faixas de idade

In [0]:
df["Faixa_Idade"] = pd.cut(
    df["Idade"],
    bins=[17, 25, 35, 50, 65, 90, float("inf")],
    labels=["18-25", "26-35", "36-50", "51-65", "66-90", "90+"],
    include_lowest=True
)

In [0]:
df["Faixa_Idade"].isna().sum()

In [0]:
df["Faixa_Idade"]. describe()

In [0]:
df_fora_regra = df[
    (df["Idade"] < 18) |
    (df["Idade"] >= 90) |
    (df["Idade"].isna())
]

print(df_fora_regra[["Data_Nascimento", "Idade"]])

In [0]:
print(df_fora_regra.shape[0])

**Qual a distribuição por idade?**

In [0]:
df["Faixa_Idade"].value_counts().sort_index()

In [0]:
age_pct = (
    df["Faixa_Idade"]
    .value_counts(normalize=True)
    .reindex(["18-25", "26-35", "36-50", "51-65", "65+"])
    * 100
)

plt.figure(figsize=(10, 6))

ax = age_pct.plot(kind="bar")

plt.title("Distribuição dos Clientes por Faixa Etária")
plt.xlabel("Faixa Etária")
plt.ylabel("Percentual (%)")
plt.xticks(rotation=0)

for i, value in enumerate(age_pct):
    ax.text(
        i,
        value,
        f"{value:.1f}%",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

In [0]:
df["Idade"].describe()

**Como os clientes estão distribuídos geograficamente?**

In [0]:
Localizacao_pct = (
    df["Localizacao"]
    .value_counts(normalize=True)
    .head(10)
    * 100
)

plt.figure(figsize=(12, 6))

ax = Localizacao_pct.plot(kind="bar")

plt.title("Top 10 Localizações dos Clientes")
plt.xlabel("Localização")
plt.ylabel("Percentual (%)")
plt.xticks(rotation=45, ha="right")

for i, value in enumerate(Localizacao_pct):
    ax.text(
        i,
        value,
        f"{value:.1f}%",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

In [0]:
%sql
select CustLocation,
COUNT(*) AS total_clientes,
ROUND((COUNT(*) * 100.0 / SUM(COUNT(*)) OVER ()), 2) AS percentual_total
from bank_transactions
group by CustLocation
order by total_clientes desc limit 10





**Existe concentração em determinadas regiões?**

In [0]:
import plotly.express as px

# Coordenadas aproximadas das principais cidades
coords = {
    "MUMBAI": [19.0760, 72.8777],
    "NEW DELHI": [28.6139, 77.2090],
    "BANGALORE": [12.9716, 77.5946],
    "GURGAON": [28.4595, 77.0266],
    "DELHI": [28.7041, 77.1025],
    "NOIDA": [28.5355, 77.3910],
    "CHENNAI": [13.0827, 80.2707],
    "PUNE": [18.5204, 73.8567],
    "HYDERABAD": [17.3850, 78.4867],
    "THANE": [19.2183, 72.9781]
}

location_df = Localizacao_pct.reset_index()
location_df.columns = ["Location", "Percentage"]

location_df["Latitude"] = location_df["Location"].map(
    lambda x: coords.get(x, [None, None])[0]
)

location_df["Longitude"] = location_df["Location"].map(
    lambda x: coords.get(x, [None, None])[1]
)

fig = px.scatter_geo(
    location_df.dropna(),
    lat="Latitude",
    lon="Longitude",
    size="Percentage",
    hover_name="Location",
    hover_data={"Percentage": ":.2f"},
    scope="asia",
    title="Concentração Geográfica dos Clientes"
)

fig.show()

As cinco principais localidades, juntas, representam aproximadamente **39,6% de toda a base de clientes**.

A distribuição é **bastante concentrada nos grandes centros urbanos da Índia**. Mumbai lidera com **9,88%** dos clientes, seguida por New Delhi (**8,10%**), Bangalore (**7,78%**), Gurgaon (**7,04%**) e Delhi (**6,77%**).

Além disso, aparecem outros importantes polos urbanos, como Noida, Chennai, Pune e Hyderabad, mas com participações individuais menores.


**Sim, existe uma concentração geográfica relevante.**

O principal destaque é o eixo **Delhi–NCR**, considerando **New Delhi, Gurgaon, Delhi, Noida e Ghaziabad**. Somando essas localidades, temos aproximadamente **26,6% da base**.

Também há uma concentração importante no entorno de **Mumbai**, considerando Mumbai, Thane e Navi Mumbai, que representam aproximadamente **13,2%**.

Isso indica que a carteira não está distribuída de maneira uniforme pelo território: existe forte presença em **clusters metropolitanos**, especialmente nas regiões de **Delhi-NCR e Mumbai**.

**Qual a distribuição por gênero?**

In [0]:
%sql
select CustGender,
COUNT(*) AS total_clientes,
ROUND((COUNT(*) * 100.0 / SUM(COUNT(*)) OVER ()), 2) AS percentual_total
from bank_transactions
group by CustGender
order by total_clientes desc 



In [0]:
df['Genero'].value_counts().sort_index()

In [0]:
CustGender_pct = (
    df["Genero"]
    .value_counts(normalize=True)
    * 100
)

plt.figure(figsize=(8, 6))
ax = CustGender_pct.plot(kind="bar")

plt.title("Distribuição dos Clientes por Gênero")
plt.xlabel("Gênero")
plt.ylabel("Percentual (%)")
plt.xticks(rotation=0)

for i, value in enumerate(CustGender_pct):
    ax.text(
        i,
        value,
        f"{value:.1f}%",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

A carteira de clientes é **predominantemente masculina**, com aproximadamente **3 em cada 4 clientes sendo homens**. As mulheres representam pouco mais de um quarto da base.

Para uma análise de negócio, essa concentração pode ser relevante para avaliar se **produtos, campanhas e estratégias de relacionamento estão adequadamente direcionados aos diferentes perfis de clientes**.

**Qual o saldo médio**

In [0]:
df["Saldo_Conta"].describe()

In [0]:
plt.figure(figsize=(8, 6))
sns.boxplot(y=df["Saldo_Conta"], color="skyblue")
plt.title("Distribuição do Saldo da Conta (Boxplot)")
plt.ylabel("Saldo da Conta (R$)")
plt.show()

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# Filtrando para pegar até o percentil 99 para melhorar a visualização do gráfico
limite_99 = df["Saldo_Conta"].quantile(0.99)
dados_filtrados = df[df["Saldo_Conta"] <= limite_99]

plt.figure(figsize=(10, 6))
sns.histplot(dados_filtrados["Saldo_Conta"], bins=50, kde=True, color="teal")
plt.title("Distribuição do Saldo da Conta (Até o Percentil 99)")
plt.xlabel("Saldo da Conta (R$)")
plt.ylabel("Quantidade de Clientes")
plt.show()

**A base é extremamente assimétrica**

A média (115,4 mil) é quase 7x maior que a mediana (16,8 mil).
**Isso mostra que poucos clientes possuem saldos muito elevados e estão puxando a média para cima.**

O cliente "típico" tem saldo bem menor
25% dos clientes: até R$ 4,7 mil
**50%: até R$ 16,8 mil**
75%: até R$ 57,7 mil
25% estão acima de R$ 57,7 mil

**Então, para representar o comportamento típico da base, R$ 16,8 mil é muito mais representativo que R$ 115,4 mil.**

In [0]:
# Quantos clientes têm saldo acima de 10 milhões?
acima_10mi = df[df["Saldo_Conta"] > 10000000]
print(f"Clientes com mais de R$ 10 milhões: {len(acima_10mi)}")

# Quantos clientes têm saldo acima de 50 milhões?
acima_50mi = df[df["Saldo_Conta"] > 50000000]
print(f"Clientes com mais de R$ 50 milhões: {len(acima_50mi)}")

In [0]:
# Mostra os 10 maiores saldos da base (junto com o ID do cliente ou índice, se houver)
display(df.nlargest(10, "Saldo_Conta"))

In [0]:
df["Saldo_Conta"].describe(percentiles=[.01, .05, .10, .25, .50, .75, .90, .95, .99])

In [0]:
df["Saldo_Conta"].nlargest(
    int(len(df) * 0.10)
).sum() / df["Saldo_Conta"].sum() * 100

**A base apresenta elevada concentração de saldo: os 10% de clientes com maiores saldos concentram aproximadamente 76,8% do saldo total, enquanto os 80% de menor saldo concentram apenas 12,3%.**

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# 1. Definir os limites das faixas de saldo (baseado na sua distribuição)
bins = [-float("inf"), 10000, 50000, 100000, 1000000, float("inf")]
labels = [
    "Até R$ 10 mil",
    "R$ 10k a R$ 50k",
    "R$ 50k a R$ 100k",
    "R$ 100k a R$ 1 Mi",
    "Acima de R$ 1 Mi",
]

# 2. Criar a coluna de faixa no DataFrame
df["Faixa_Saldo"] = pd.cut(
    df["Saldo_Conta"], bins=bins, labels=labels, include_lowest=True
)

# 3. Calcular a quantidade de clientes e o saldo total por faixa
resumo_faixas = (
    df.groupby("Faixa_Saldo", observed=False)
    .agg(
        Qtd_Clientes=("Saldo_Conta", "count"),
        Saldo_Total=("Saldo_Conta", "sum"),
    )
    .reset_index()
)

# Calcular as porcentagens
resurso_faixas = resumo_faixas.copy()
resumo_faixas["Pct_Clientes"] = (
    resumo_faixas["Qtd_Clientes"] / resumo_faixas["Qtd_Clientes"].sum()
) * 100
resumo_faixas["Pct_Saldo_Total"] = (
    resumo_faixas["Saldo_Total"] / resumo_faixas["Saldo_Total"].sum()
) * 100

display(resumo_faixas)

# 4. Gerar o Gráfico de Barras Comparativo
fig, ax1 = plt.subplots(figsize=(10, 6))

x = range(len(resumo_faixas))
width = 0.35

rects1 = ax1.bar(
    [p - width / 2 for p in x],
    resumo_faixas["Pct_Clientes"],
    width,
    label="% de Clientes",
    color="#4c72b0",
)
rects2 = ax1.bar(
    [p + width / 2 for p in x],
    resumo_faixas["Pct_Saldo_Total"],
    width,
    label="% do Saldo Total",
    color="#55a868",
)

ax1.set_ylabel("Participação (%)")
ax1.set_title("Comparativo: % de Clientes vs % do Saldo Total por Faixa")
ax1.set_xticks(x)
ax1.set_xticklabels(resumo_faixas["Faixa_Saldo"])
ax1.legend()

plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

A base da pirâmide (Até R$ 50 mil):

Clientes: Somando as duas primeiras faixas ("Até R$ 10 mil" e "R$ 10k a R$ 50k"), temos mais de 72% de todos os clientes da base (38,1% + 34,5%).

O dinheiro: No entanto, toda essa massa gigante de pessoas detém apenas cerca de 8,3% do saldo total da empresa (1,1% + 7,1%).

A classe média/alta (R$ 50k a R$ 1 Mi):

As faixas intermediárias concentram uma parcela equilibrada, mas o destaque vai para a faixa de R$ 100k a R$ 1 Mi, que reúne 15,2% dos clientes e acumula 38,2% de todo o dinheiro.

O topo da pirâmide (Acima de R$ 1 Mi):

Clientes: Apenas 1,8% dos clientes estão na faixa milionária (acima de R$ 1 milhão).

O dinheiro: Sozinho, esse grupo minúsculo concentra 47,2% de todo o saldo da base.

**Conclusão para o estudo de caso:**
Esse gráfico é a tradução visual da famosa Regra de Pareto (80/20) ou de uma distribuição altamente concentrada. Ele prova estatisticamente que a estratégia de negócios da instituição precisa ser dividida:

Focar em escala e volume para atender bem à massa (que traz grande capilaridade de clientes).

Focar em retenção e atendimento personalizado (Private/High Net Worth) para cuidar daquele pequeno grupo de 1,8% que segura quase metade de todo o capital da empresa.

**Qual o valor médio das transações?**

In [0]:
df["Valor_Transacao"].describe()

valor das transações apresenta forte concentração em valores baixos: 50% das operações são de até 459, enquanto a média sobe para 1.574 devido à presença de transações de valor muito elevado.

**Que tipo de transação é mais comum?**

In [0]:
df["Faixa_Transacao"] = pd.cut(
    df["Valor_Transacao"],
    bins=[-1, 100, 500, 1000, 5000, 10000, float("inf")],
    labels=[
        "Até R$ 100",
        "R$ 101–500",
        "R$ 501–1.000",
        "R$ 1.001–5.000",
        "R$ 5.001–10.000",
        "Acima de R$ 10.000"
    ]
)

In [0]:
df["Faixa_Transacao"].describe()

In [0]:
transacao_pct = (
    df["Faixa_Transacao"]
    .value_counts(normalize=True)
    .sort_index()
    * 100
)

transacao_pct

In [0]:
plt.figure(figsize=(10, 6))

ax = transacao_pct.plot(kind="bar")

plt.title("Distribuição das Transações por Faixa de Valor")
plt.xlabel("Faixa de Transação")
plt.ylabel("Percentual das Transações (%)")
plt.xticks(rotation=0)

# Valores acima das barras
for i, value in enumerate(transacao_pct):
    ax.text(
        i,
        value + 0.5,
        f"{value:.1f}%",
        ha="center"
    )

plt.tight_layout()
plt.show()

75% das transações, por exemplo, estão abaixo de R$ 1.000.

Isso mostra que o comportamento transacional é predominantemente de baixo/médio valor, mesmo que existam algumas operações extremamente grandes.

In [0]:
df["Valor_Transacao"].nlargest(
    int(len(df) * 0.20)
).sum() / df["Valor_Transacao"].sum() * 100

In [0]:
qtd_pct = (
    (df["Valor_Transacao"] > 10000).sum()
    / len(df)
    * 100
)

qtd_pct

O volume financeiro está fortemente concentrado nas maiores operações: as 20% maiores transações respondem por 78,1% de todo o valor movimentado, enquanto apenas 2,73% das operações superam R$ 10 mil. Esse cenário indica que uma parcela reduzida de operações possui peso desproporcional no resultado financeiro, tornando essencial identificar quais clientes e perfis estão por trás dessas transações de alto valor.

**quantas transações, em média, cada cliente realizou.``**

In [0]:
# Quantidade de transações realizadas por cliente
transacoes_por_cliente = (
    df.groupby("ID_Cliente")["ID_Transacao"]
    .count()
)

frequencia_media = transacoes_por_cliente.mean()
frequencia_mediana = transacoes_por_cliente.median()

print(f"Frequência média: {frequencia_media:.2f} transações por cliente")
print(f"Frequência mediana: {frequencia_mediana:.0f} transações por cliente")

In [0]:
df_frequencia = transacoes_por_cliente.reset_index(
    name="Qtd_Transacoes"
)

df_frequencia["Faixa_Frequencia"] = pd.cut(
    df_frequencia["Qtd_Transacoes"],
    bins=[0, 1, 2, 5, 10, 20, float("inf")],
    labels=[
        "1 transação",
        "2 transações",
        "3–5 transações",
        "6–10 transações",
        "11–20 transações",
        "Acima de 20"
    ]
)

frequencia_pct = (
    df_frequencia["Faixa_Frequencia"]
    .value_counts(normalize=True)
    .sort_index()
    * 100
)
plt.figure(figsize=(10, 6))

ax = frequencia_pct.plot(kind="bar")

plt.title("Distribuição dos Clientes por Frequência de Utilização")
plt.xlabel("Quantidade de Transações")
plt.ylabel("Percentual de Clientes (%)")
plt.xticks(rotation=0)

for i, value in enumerate(frequencia_pct):
    ax.text(
        i,
        value + 0.5,
        f"{value:.1f}%",
        ha="center"
    )

plt.tight_layout()
plt.show()
frequencia_pct

In [0]:
df_frequencia["Faixa_Frequencia"].describe()

A base apresenta baixa frequência de relacionamento transacional: 97,9% dos clientes realizaram no máximo duas transações, enquanto apenas 2,1% apresentaram recorrência de 3 a 5 operações. Isso sugere que o volume financeiro observado está concentrado em poucas operações, e não em uma alta frequência de uso por cliente.

**“A carteira é predominantemente composta por homens, com idade média de 41 anos, concentrada nos grandes centros urbanos, especialmente Delhi-NCR e Mumbai. As transações apresentam valor médio de R$ 1.574, porém a mediana é de apenas R$ 459, evidenciando forte assimetria causada por poucas operações de alto valor. A utilização também é pouco frequente: 97,9% dos clientes realizam no máximo duas transações.”**

Para o seu projeto

Isso é importante porque você começa a enxergar diferentes perfis de valor na carteira:

1. Alta frequência / baixo valor
Clientes que realizam muitas transações, mas de valores menores.

2. Baixa frequência / alto valor
Poucas transações, porém com grande impacto no volume financeiro.

Isso abre uma oportunidade para a próxima etapa do projeto: identificar quem são os clientes responsáveis por esse alto volume financeiro.

Uma boa pergunta para continuar seria:

Quem são os clientes responsáveis pela maior parte do volume financeiro do banco?

Aí você pode cruzar Cliente × quantidade de transações × valor total transacionado × ticket médio. Esse cruzamento começa a transformar o diagnóstico da carteira em uma análise realmente comercial.